In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier 


pd.set_option('display.max_columns', None)   # 모든 컬럼 표시
pd.set_option('display.width', None)         # 줄바꿈 없이 전체 폭 사용
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 생략 안 함print(df)

df = pd.read_csv('data/steam_reviews_random50k.csv')
df.head()

,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,17985344,644930,They Are Billions,66899072,schinese,666,1586361984,1586361984,True,0,0,0.000000,0,True,False,False,76561198237086364,31,6,642.0,0.0,194.0,1.591716e+09
1,15288688,578080,PLAYERUNKNOWN'S BATTLEGROUNDS,34432737,english,21/10 would play again but f$ck the hackers and anti-cheat,1503476147,1510637434,True,0,0,0.474597,0,False,False,True,76561198340731525,21,2,7467.0,0.0,2639.0,1.566988e+09
2,20773381,671510,Desolate,49320648,russian,"Есть баги, причем жесткие, у меня например не прогружается квестовое место и игра пока стоит, жду реакции разработчиков, но в общем не плохая игра.",1551552086,1551552086,True,0,0,0.000000,0,True,False,False,76561198379377765,94,8,1191.0,0.0,1191.0,1.552250e+09
3,18067011,322330,Don't Starve Together,72177433,tchinese,←MOD愛好者（沒錯就是菜了）\n認真玩不加MOD會很有挑戰性\n加各種MOD的話就會是“相對的”休閑游戲\n單機/聯機都很快樂 和朋友一起更快樂w\n光是活著就可以很殺時間了w\n經常一玩一整夜 妥妥的殺時間利器\n而且工作坊也有很多其他大大做的角色（就算沒錢買部分新角色也不要緊！\n總之買就對啦！\n\n——啊記得吃東西，你的角色說他餓啦,1594057760,1594057760,True,0,0,0.000000,0,False,False,False,76561198997362162,21,9,4933.0,0.0,3257.0,1.609804e+09
4,18784582,105600,Terraria,50790126,russian,Просто супер . Игра класс . Геймплей тОп,1558643438,1558643438,True,0,0,0.000000,0,True,False,False,76561198938103721,2,2,82.0,0.0,24.0,1.584604e+09


| 한글 컬럼명 | 영문 컬럼명 | 설명(내용) | 범위(실제 분포 기준) |
|---|---|---|---|
| 앱 ID | app_id | Steam 앱 고유 식별자 | 정수 (수천만 단위, 예: 43 ~ 85,000,000+) |
| 앱 이름 | app_name | 게임 또는 앱 이름 | 문자열 |
| 리뷰 ID | review_id | 리뷰 고유 식별자 | 정수 (고유값) |
| 리뷰 언어 | language | 리뷰가 작성된 언어 | 문자열 (예: english, schinese 등) |
| 리뷰 내용 | review | 리뷰 텍스트 본문 | 문자열 (길이 가변) |
| 리뷰 생성 시각 | timestamp_created | 리뷰 작성 시각 | Unix Timestamp (약 1.29B ~ 1.61B) |
| 리뷰 수정 시각 | timestamp_updated | 리뷰 마지막 수정 시각 | Unix Timestamp (약 1.29B ~ 2.28B) |
| 추천 여부 | recommended | 게임 추천 여부 | Boolean (true / false, true ≈ 87%) |
| 도움됨 투표 수 | votes_helpful | 도움됨(Helpful) 투표 수 | 정수 (0 ~ 약 4,900) |
| 재미있음 투표 수 | votes_funny | 재미있음(Funny) 투표 수 | 정수 (0 ~ 약 27,000) |
| 가중 투표 점수 | weighted_vote_score | 도움됨 기반 가중 점수 | 실수 (0.0 ~ 1.0) |
| 댓글 수 | comment_count | 리뷰 댓글 수 | 정수 (0 ~ 약 1,300,000) |
| 스팀 구매 여부 | steam_purchase | Steam에서 직접 구매했는지 여부 | Boolean (true ≈ 77%) |
| 무료 획득 여부 | received_for_free | 무료 획득 여부 | Boolean (true ≈ 3%) |
| 얼리액세스 리뷰 | written_during_early_access | 얼리 액세스 중 작성 여부 | Boolean (true ≈ 9%) |
| 작성자 SteamID | author.steamid | 리뷰 작성자 SteamID | 64-bit 정수 (약 7.6e16 ~ 7.7e16) |
| 작성자 보유 게임 수 | author.num_games_owned | 보유 게임 개수 | 정수 (0 ~ 약 21,700,000) |
| 작성자 리뷰 수 | author.num_reviews | 작성자 전체 리뷰 수 | 정수 (0 ~ 약 1,290,000) |
| 누적 플레이 시간 | author.playtime_forever | 총 플레이 시간 | 초 단위 정수 (0 ~ 약 85,000,000초) |
| 최근 2주 플레이 시간 | author.playtime_last_two_weeks | 최근 2주 플레이 시간 | 초 단위 정수 (0 ~ 약 3,700,000초) |
| 리뷰 시점 플레이 시간 | author.playtime_at_review | 리뷰 당시 플레이 시간 | 초 단위 정수 (0 ~ 약 27,000초) |
| 마지막 플레이 시각 | author.last_played | 마지막 플레이 시각 | Unix Timestamp (약 1.29B ~ 1.61B) |


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 23 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Unnamed: 0                      50000 non-null  int64  
 1   app_id                          50000 non-null  int64  
 2   app_name                        50000 non-null  object 
 3   review_id                       50000 non-null  int64  
 4   language                        50000 non-null  object 
 5   review                          49927 non-null  object 
 6   timestamp_created               50000 non-null  int64  
 7   timestamp_updated               50000 non-null  int64  
 8   recommended                     50000 non-null  bool   
 9   votes_helpful                   50000 non-null  int64  
 10  votes_funny                     50000 non-null  int64  
 11  weighted_vote_score             50000 non-null  float64
 12  comment_count                   

In [6]:
df.isnull().sum()

Unnamed: 0                         0
app_id                             0
app_name                           0
review_id                          0
language                           0
review                            73
timestamp_created                  0
timestamp_updated                  0
recommended                        0
votes_helpful                      0
votes_funny                        0
weighted_vote_score                0
comment_count                      0
steam_purchase                     0
received_for_free                  0
written_during_early_access        0
author.steamid                     0
author.num_games_owned             0
author.num_reviews                 0
author.playtime_forever            0
author.playtime_last_two_weeks     0
author.playtime_at_review         61
author.last_played                 0
dtype: int64

In [10]:
df = df.dropna()
df.isnull().sum()

Unnamed: 0                        0
app_id                            0
app_name                          0
review_id                         0
language                          0
review                            0
timestamp_created                 0
timestamp_updated                 0
recommended                       0
votes_helpful                     0
votes_funny                       0
weighted_vote_score               0
comment_count                     0
steam_purchase                    0
received_for_free                 0
written_during_early_access       0
author.steamid                    0
author.num_games_owned            0
author.num_reviews                0
author.playtime_forever           0
author.playtime_last_two_weeks    0
author.playtime_at_review         0
author.last_played                0
dtype: int64

In [11]:
df.describe()

,Unnamed: 0,app_id,review_id,timestamp_created,timestamp_updated,votes_helpful,votes_funny,weighted_vote_score,comment_count,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
count,4.986600e+04,4.986600e+04,4.986600e+04,4.986600e+04,4.986600e+04,49866.000000,49866.000000,49866.000000,49866.000000,4.986600e+04,49866.000000,49866.000000,4.986600e+04,49866.000000,4.986600e+04,4.986600e+04
mean,1.084944e+07,3.946838e+05,5.198097e+07,1.544704e+09,1.547863e+09,2.103658,0.938796,0.166532,0.140456,7.656120e+16,132.578470,10.702964,1.612663e+04,160.297618,8.691323e+03,1.580601e+09
std,6.309651e+06,2.503107e+05,2.082557e+07,5.801955e+07,5.673656e+07,50.056597,44.725798,0.243811,2.161198,3.168252e+08,264.400046,38.790147,3.765936e+04,779.629080,2.348173e+04,4.747525e+07
min,4.000000e+00,7.000000e+01,1.491300e+04,1.290222e+09,1.290649e+09,0.000000,0.000000,0.000000,0.000000,7.656120e+16,0.000000,1.000000,5.000000e+00,0.000000,1.000000e+00,8.640000e+04
25%,5.360368e+06,2.427600e+05,3.651037e+07,1.510663e+09,1.511865e+09,0.000000,0.000000,0.000000,0.000000,7.656120e+16,22.000000,2.000000,1.239000e+03,0.000000,5.510000e+02,1.573667e+09
50%,1.085520e+07,3.595500e+05,5.392164e+07,1.562508e+09,1.572513e+09,0.000000,0.000000,0.000000,0.000000,7.656120e+16,61.000000,4.000000,4.296000e+03,0.000000,1.873000e+03,1.599000e+09
75%,1.631056e+07,5.780800e+05,6.943405e+07,1.589912e+09,1.591622e+09,1.000000,0.000000,0.485853,0.000000,7.656120e+16,146.000000,10.000000,1.493775e+04,0.000000,6.791000e+03,1.608959e+09
max,2.174732e+07,1.291340e+06,8.521524e+07,1.611422e+09,1.611422e+09,9307.000000,8983.000000,0.989001,238.000000,7.656120e+16,13798.000000,2834.000000,1.763601e+06,20510.000000,1.282544e+06,1.611429e+09


In [12]:
df.columns

Index(['Unnamed: 0', 'app_id', 'app_name', 'review_id', 'language', 'review',
       'timestamp_created', 'timestamp_updated', 'recommended',
       'votes_helpful', 'votes_funny', 'weighted_vote_score', 'comment_count',
       'steam_purchase', 'received_for_free', 'written_during_early_access',
       'author.steamid', 'author.num_games_owned', 'author.num_reviews',
       'author.playtime_forever', 'author.playtime_last_two_weeks',
       'author.playtime_at_review', 'author.last_played'],
      dtype='object')

In [14]:
# 도메인으로 필터링 함
drop_columns = [
    'Unnamed: 0',
    'app_id',
    'app_name',
    'review_id',
]
df = df.drop(columns=drop_columns)
df

,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,schinese,666,1586361984,1586361984,True,0,0,0.000000,0,True,False,False,76561198237086364,31,6,642.0,0.0,194.0,1.591716e+09
1,english,21/10 would play again but f$ck the hackers and anti-cheat,1503476147,1510637434,True,0,0,0.474597,0,False,False,True,76561198340731525,21,2,7467.0,0.0,2639.0,1.566988e+09
2,russian,"Есть баги, причем жесткие, у меня например не прогружается квестовое место и игра пока стоит, жду реакции разработчиков, но в общем не плохая игра.",1551552086,1551552086,True,0,0,0.000000,0,True,False,False,76561198379377765,94,8,1191.0,0.0,1191.0,1.552250e+09
3,tchinese,←MOD愛好者（沒錯就是菜了）\n認真玩不加MOD會很有挑戰性\n加各種MOD的話就會是“相對的”休閑游戲\n單機/聯機都很快樂 和朋友一起更快樂w\n光是活著就可以很殺時間了w\n經常一玩一整夜 妥妥的殺時間利器\n而且工作坊也有很多其他大大做的角色（就算沒錢買部分新角色也不要緊！\n總之買就對啦！\n\n——啊記得吃東西，你的角色說他餓啦,1594057760,1594057760,True,0,0,0.000000,0,False,False,False,76561198997362162,21,9,4933.0,0.0,3257.0,1.609804e+09
4,russian,Просто супер . Игра класс . Геймплей тОп,1558643438,1558643438,True,0,0,0.000000,0,True,False,False,76561198938103721,2,2,82.0,0.0,24.0,1.584604e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,english,This is my Fav,1582923141,1582923141,True,0,0,0.000000,0,True,False,False,76561198289018357,47,2,5569.0,13.0,5311.0,1.610241e+09
49996,polish,Dzięki Epic,1600520618,1600520618,False,1,0,0.000000,0,False,False,False,76561198026111891,170,2,70456.0,647.0,64864.0,1.610992e+09
49997,danish,.,1574975203,1574975203,True,0,0,0.000000,0,True,False,False,76561198981747323,6,3,834.0,0.0,319.0,1.608934e+09
49998,english,"It's fun. I stabbed a bear while b-hopping around it like ADHD is going out of style and I just threw out all my Ritalin. Solid meme, you can build towers to stab flying birds with spears. There are some strange rat-creatures? They look hideous and taste worse. I played with a friend who was really bad though so that somewhat injured my experience. At one point, the idiot died and I died trying to save him so I had to use a second alt account to revive both of us. Too much effort definitely, so if you have friends, don't play with them, or use an easier difficulty. Also, turn on friendly fire, nothing could go wrong. I solidly recommend to anyone who enjoys the finer things in life.\n\nUpdate: I've discovered my friend was high while playing. It's not as difficult as he implied.",1578277979,1578278158,True,1,0,0.523810,5,True,False,True,76561198181058465,211,9,1008.0,0.0,966.0,1.578366e+09


In [23]:
df[df['timestamp_updated'].max() - df['author.last_played'] > 7776000]

,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,schinese,666,1586361984,1586361984,True,0,0,0.000000,0,True,False,False,76561198237086364,31,6,642.0,0.0,194.0,1.591716e+09
1,english,21/10 would play again but f$ck the hackers and anti-cheat,1503476147,1510637434,True,0,0,0.474597,0,False,False,True,76561198340731525,21,2,7467.0,0.0,2639.0,1.566988e+09
2,russian,"Есть баги, причем жесткие, у меня например не прогружается квестовое место и игра пока стоит, жду реакции разработчиков, но в общем не плохая игра.",1551552086,1551552086,True,0,0,0.000000,0,True,False,False,76561198379377765,94,8,1191.0,0.0,1191.0,1.552250e+09
4,russian,Просто супер . Игра класс . Геймплей тОп,1558643438,1558643438,True,0,0,0.000000,0,True,False,False,76561198938103721,2,2,82.0,0.0,24.0,1.584604e+09
7,schinese,就是坨屎联机根本连不上,1534336522,1534336522,False,0,0,0.000000,0,False,False,False,76561198253137205,15,2,7475.0,0.0,4745.0,1.590500e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49986,brazilian,É SEM DÚVIDAS O MELHOR JOGO DA ATUALIDADE !! =D,1439733033,1439733033,True,2,0,0.545455,0,True,False,False,76561198103754795,82,42,4050.0,0.0,2854.0,1.459901e+09
49989,english,they should ban this game in china,1511459248,1511459248,True,3,0,0.547989,0,False,False,True,76561198132487673,92,3,24175.0,0.0,17153.0,1.574648e+09
49994,english,"After all these years they still can't get the ""auto"" part in Grand Theft Auto correctly. 'All you had to do, was save the car after the mission CJ.'\n\nGarage full of custom cars, can never use them because the minute I start a mission it's gone. \n\nThe three character system also meant that everytime you started a mission, you had to resupply three times. And with trevor always lying in a field in the middle of nowhere, his character trait stopped being funny and became annoying.\n\nDid not thorougly enjoy this one, and given it's budget and hype (which I steered clear of) i'd say that's condeming enough.",1483737823,1483737823,False,1,0,0.523810,0,True,False,False,76561197979785822,274,49,10512.0,0.0,4265.0,1.558549e+09
49998,english,"It's fun. I stabbed a bear while b-hopping around it like ADHD is going out of style and I just threw out all my Ritalin. Solid meme, you can build towers to stab flying birds with spears. There are some strange rat-creatures? They look hideous and taste worse. I played with a friend who was really bad though so that somewhat injured my experience. At one point, the idiot died and I died trying to save him so I had to use a second alt account to revive both of us. Too much effort definitely, so if you have friends, don't play with them, or use an easier difficulty. Also, turn on friendly fire, nothing could go wrong. I solidly recommend to anyone who enjoys the finer things in life.\n\nUpdate: I've discovered my friend was high while playing. It's not as difficult as he implied.",1578277979,1578278158,True,1,0,0.523810,5,True,False,True,76561198181058465,211,9,1008.0,0.0,966.0,1.578366e+09


# 기준일만으로 세달전을 기준으로 했음에도 이탈률로 정하기에는 너무 많다
> 해당 유저